# Air Quality PySpark & Databricks Analytics Project
### Dataset: UCI Air Quality Multisensor Array
**Dataset Link:** [UCI Air Quality Dataset #360](https://archive.ics.uci.edu/dataset/360/air+quality)

---
### Project Milestones & Video Timestamps:
- **[4:12] Dataset Load:** Reading European CSV with semicolon (`;`) delimiter & custom schema mapping.
- **[6:17] PySpark DataFrame:** Missing value handling (`-200` to `NULL`), comma-decimal casting, timestamp extraction, and feature engineering.
- **[18:55] Project Implementation and KPI Development:**
  - **Objective 1:** Diurnal Rush-Hour Patterns & Weekend Emission Abatement.
  - **Objective 2:** Low-Cost Chemical Sensor Calibration vs. Reference Ground Truth.
  - **Objective 3:** Environmental Health Compliance & EU Regulatory Limit Violations.
- **[41:15] Dashboard Visualization:** Interactive visual plots, trend curves, correlation scatter plots, and pollution heatmaps in VS Code.

In [ ]:
# 1. Initialize PySpark Environment
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, to_timestamp, concat, lit, regexp_replace, hour, dayofweek, month, avg, count, round as sp_round

os.environ.pop("SPARK_HOME", None)  # Prevent Windows version conflicts
spark = SparkSession.builder.appName("AirQuality_VSCode").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark Session Active: Version {spark.version}")

## [4:12] Milestone 1: Dataset Load
Ingest raw CSV file with semicolon delimiter and map the 15 valid environmental attributes.

In [ ]:
data_path = "../data/AirQualityUCI.csv"
raw_df = spark.read.option("header", "true").option("delimiter", ";").csv(data_path)

clean_names = [
    "date_str", "time_str", "co_gt", "sensor_s1_co", "nmhc_gt", "c6h6_gt",
    "sensor_s2_nmhc", "nox_gt", "sensor_s3_nox", "no2_gt", "sensor_s4_no2",
    "sensor_s5_o3", "temperature", "relative_humidity", "absolute_humidity"
]

dummy_cols = [f"c_{i}" for i in range(len(raw_df.columns))]
for i, name in enumerate(clean_names):
    dummy_cols[i] = name
df_mapped = raw_df.toDF(*dummy_cols).select(*clean_names)
print(f"Raw records loaded: {df_mapped.count():,}")

## [6:17] Milestone 2: PySpark DataFrame Cleaning & Feature Engineering
- Convert European decimal commas (`,`) to dots (`.`)
- Filter out blank/null dates
- Cast numeric columns to `DoubleType`
- Replace `-200` sentinel with Spark `NULL`
- Synthesize `timestamp` and derive `hour`, `month`, `day_of_week`, `is_weekend`, `is_rush_hour`

In [ ]:
df_clean = df_mapped.filter(col("date_str").isNotNull() & (col("date_str") != ""))

decimal_cols = ["co_gt", "c6h6_gt", "temperature", "relative_humidity", "absolute_humidity"]
for c in decimal_cols:
    df_clean = df_clean.withColumn(c, regexp_replace(col(c), ",", ".").cast("double"))

sensor_cols = ["sensor_s1_co", "nmhc_gt", "sensor_s2_nmhc", "nox_gt", "sensor_s3_nox", "no2_gt", "sensor_s4_no2", "sensor_s5_o3"]
for c in sensor_cols:
    df_clean = df_clean.withColumn(c, col(c).cast("double"))

for c in decimal_cols + sensor_cols:
    df_clean = df_clean.withColumn(c, when(col(c) == -200, None).otherwise(col(c)))

df_curated = (
    df_clean
    .withColumn("timestamp", to_timestamp(concat(col("date_str"), lit(" "), col("time_str")), "dd/MM/yyyy HH.mm.ss"))
    .withColumn("hour", hour(col("timestamp")))
    .withColumn("month", month(col("timestamp")))
    .withColumn("day_of_week", dayofweek(col("timestamp")))
    .withColumn("is_weekend", when(col("day_of_week").isin(1, 7), 1).otherwise(0))
    .withColumn("is_rush_hour", when(col("hour").isin(8, 9, 18, 19, 20), 1).otherwise(0))
)
df_curated.createOrReplaceTempView("air_quality_curated")
print(f"Curated valid hourly records: {df_curated.count():,}")
df_curated.select("timestamp", "co_gt", "c6h6_gt", "nox_gt", "sensor_s1_co", "temperature", "is_rush_hour").show(5)

## [18:55] Milestone 3: Project Implementation & KPI Development
### Objective 1: Diurnal Rush-Hour Patterns & Weekend Traffic Abatement

In [ ]:
rush_kpi = df_curated.groupBy("is_rush_hour").agg(
    sp_round(avg("co_gt"), 2).alias("avg_co_mg_m3"),
    sp_round(avg("c6h6_gt"), 2).alias("avg_benzene_ug_m3"),
    sp_round(avg("nox_gt"), 2).alias("avg_nox_ppb"),
    sp_round(avg("no2_gt"), 2).alias("avg_no2_ug_m3"),
    count("timestamp").alias("total_hours")
)
rush_kpi.show()

### Objective 2: Sensor Calibration & Cross-Correlation Matrix

In [ ]:
r_co = df_curated.stat.corr("sensor_s1_co", "co_gt")
r_benzene = df_curated.stat.corr("sensor_s2_nmhc", "c6h6_gt")
r_nox = df_curated.stat.corr("sensor_s3_nox", "nox_gt")
r_no2 = df_curated.stat.corr("sensor_s4_no2", "no2_gt")

print(f"Correlation PT08.S1 (CO sensor) vs Ground Truth CO:        r = {r_co:.4f}")
print(f"Correlation PT08.S2 (NMHC sensor) vs Ground Truth Benzene:   r = {r_benzene:.4f} (High Fidelity)")
print(f"Correlation PT08.S3 (NOx sensor) vs Ground Truth NOx:        r = {r_nox:.4f} (Inverted Resistance)")
print(f"Correlation PT08.S4 (NO2 sensor) vs Ground Truth NO2:        r = {r_no2:.4f}")

### Objective 3: European Union Safety Standard Exceedance Monitoring
- EU Directive 2008/50/EC hourly NO2 threshold: **200 µg/m³**
- EU/WHO maximum daily 8-hour CO limit: **10 mg/m³**

In [ ]:
compliance_df = df_curated.agg(
    count("no2_gt").alias("monitored_hours"),
    count(when(col("no2_gt") > 200.0, 1)).alias("no2_exceedances"),
    sp_round(count(when(col("no2_gt") > 200.0, 1)) * 100.0 / count("no2_gt"), 2).alias("no2_exceedance_pct"),
    count(when(col("co_gt") > 10.0, 1)).alias("co_severe_exceedances")
)
compliance_df.show()

## [41:15] Milestone 4: Visualizations in VS Code
Render rich, interactive publication-quality charts directly inside this VS Code notebook.

In [ ]:
# Visualization 1: 24-Hour Diurnal Multi-Pollutant Dynamics
sns.set_theme(style="whitegrid")
hourly_pdf = df_curated.groupBy("hour").agg(
    avg("co_gt").alias("co"),
    avg("c6h6_gt").alias("benzene"),
    (avg("nox_gt") / 20.0).alias("nox_scaled")
).orderBy("hour").toPandas()

plt.figure(figsize=(11, 5.5))
plt.plot(hourly_pdf["hour"], hourly_pdf["co"], marker="o", linewidth=2.5, color="#d62728", label="CO (mg/m³)")
plt.plot(hourly_pdf["hour"], hourly_pdf["benzene"], marker="s", linewidth=2.5, color="#1f77b4", label="Benzene (µg/m³)")
plt.plot(hourly_pdf["hour"], hourly_pdf["nox_scaled"], marker="^", linewidth=2.5, color="#2ca02c", linestyle="--", label="NOx / 20 (ppb scaled)")
plt.axvspan(7.5, 9.5, color="#ffbb78", alpha=0.3, label="Morning Rush Hour (8-9 AM)")
plt.axvspan(17.5, 20.5, color="#ff9896", alpha=0.3, label="Evening Rush Hour (6-8 PM)")
plt.title("Objective 1: 24-Hour Diurnal Multi-Pollutant Dynamics & Commuter Peaks", fontsize=13, fontweight="bold")
plt.xlabel("Hour of Day (00:00 - 23:00)")
plt.ylabel("Average Pollutant Concentration")
plt.xticks(range(0, 24))
plt.legend()
plt.show()

In [ ]:
# Visualization 2: Sensor Calibration vs Reference Ground Truth
sample_pdf = df_curated.select("sensor_s2_nmhc", "c6h6_gt", "sensor_s1_co", "co_gt").dropna().sample(fraction=0.3, seed=42).toPandas()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
sns.regplot(data=sample_pdf, x="sensor_s2_nmhc", y="c6h6_gt", ax=ax1, color="#1f77b4", scatter_kws={"alpha": 0.35, "s": 15}, line_kws={"color": "#d62728", "linewidth": 2})
ax1.set_title(f"Sensor PT08.S2 vs Benzene C6H6(GT)\nr = {r_benzene:.3f} (High Accuracy)", fontweight="bold")
sns.regplot(data=sample_pdf, x="sensor_s1_co", y="co_gt", ax=ax2, color="#2ca02c", scatter_kws={"alpha": 0.35, "s": 15}, line_kws={"color": "#ff7f0e", "linewidth": 2})
ax2.set_title(f"Sensor PT08.S1 vs CO(GT)\nr = {r_co:.3f} (Cross-Sensitivity)", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 3: Weekly Day-vs-Hour NO2 Pollution Heatmap
heat_pdf = df_curated.groupBy("day_of_week", "hour").agg(sp_round(avg("no2_gt"), 1).alias("avg_no2")).toPandas()
day_map = {1: "Sun", 2: "Mon", 3: "Tue", 4: "Wed", 5: "Thu", 6: "Fri", 7: "Sat"}
heat_pdf["day_str"] = heat_pdf["day_of_week"].map(day_map)
pivot = heat_pdf.pivot(index="day_str", columns="hour", values="avg_no2").reindex(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
plt.figure(figsize=(12, 4.5))
sns.heatmap(pivot, cmap="YlOrRd", annot=False, cbar_kws={"label": "Mean NO2 (µg/m³)"})
plt.title("Objective 3: Temporal Pollution Heatmap (Day of Week vs. Hour of Day)", fontsize=13, fontweight="bold")
plt.xlabel("Hour of the Day (0-23)")
plt.ylabel("Day of the Week")
plt.show()